# Data Visualization for Statistics
**Topic:** Descriptive Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Select** the appropriate chart type for a given statistical question
- **Explain** what a box plot reveals that a histogram hides, and vice versa
- **Interpret** a correlation heatmap and identify the most strongly related variable pairs

> **Tip:** Start by examining the **box plot panel**, try to read the median, IQR, and outliers before looking at the corresponding histogram. See if your intuition matches.

---
## How we got here

In *04–07* we calculated summary statistics: mean, standard deviation, skewness, and correlation. Numbers are precise, but a well-chosen chart reveals structure that numbers alone miss, a bimodal distribution looks perfectly normal if you only read its mean and standard deviation. This notebook connects every statistic you have learned to a visual that makes it visible.

---
## Why this matters for data science

Every data science workflow begins with exploratory data analysis (EDA), and EDA is almost entirely visual. You will use histograms to spot skew before choosing transformations, scatter plots to detect correlation before building regression models, and box plots to compare group distributions before running hypothesis tests. The charts in this notebook are the ones you will reach for every single day.

---
## Try it yourself

In [2]:
from sklearn.datasets import load_iris

# ── Data ──────────────────────────────────────────────────────────────────────
_iris = load_iris(as_frame=True)
_iris_df = _iris.frame
_iris_df["species"] = _iris_df["target"].map(dict(enumerate(_iris.target_names)))

COLS = {
    "Sepal Length": "sepal length (cm)",
    "Sepal Width":  "sepal width (cm)",
    "Petal Length": "petal length (cm)",
    "Petal Width":  "petal width (cm)",
}
_species_list = sorted(_iris_df["species"].unique())
_sp_colors = {
    _species_list[0]: PALETTE["primary"],
    _species_list[1]: PALETTE["secondary"],
    _species_list[2]: PALETTE["muted"],
}

# ── Controls ──────────────────────────────────────────────────────────────────
out = Output()

_ctrl_w = widgets.Layout(width="260px")
_lbl_w  = {"description_width": "110px"}

var1_dd = Dropdown(
    options=list(COLS.keys()),
    value="Sepal Length",
    description="Variable:",
    style=_lbl_w,
    layout=_ctrl_w,
)
chart_dd = Dropdown(
    options=["Histogram", "Box Plot", "Violin Plot", "Scatter"],
    value="Histogram",
    description="Chart Type:",
    style=_lbl_w,
    layout=_ctrl_w,
)
species_dd = Dropdown(
    options=["All Species", "Split by Species"],
    value="All Species",
    description="Species:",
    style=_lbl_w,
    layout=_ctrl_w,
)
var2_dd = Dropdown(
    options=list(COLS.keys()),
    value="Petal Length",
    description="Y-axis:",
    style=_lbl_w,
    layout=widgets.Layout(width="260px", display="none"),
)

# ── Render ────────────────────────────────────────────────────────────────────
def render(change=None):
    col1  = COLS[var1_dd.value]
    col2  = COLS[var2_dd.value]
    chart = chart_dd.value
    split = species_dd.value == "Split by Species"

    traces = []

    if chart == "Histogram":
        if split:
            for sp in _species_list:
                traces.append(go.Histogram(
                    x=_iris_df[_iris_df["species"] == sp][col1],
                    name=sp, marker_color=_sp_colors[sp], opacity=0.6, nbinsx=20,
                ))
        else:
            traces.append(go.Histogram(
                x=_iris_df[col1], name=var1_dd.value,
                marker_color=PALETTE["primary"], nbinsx=20,
            ))

    elif chart == "Box Plot":
        if split:
            for sp in _species_list:
                traces.append(go.Box(
                    y=_iris_df[_iris_df["species"] == sp][col1],
                    name=sp, marker_color=_sp_colors[sp],
                ))
        else:
            traces.append(go.Box(
                y=_iris_df[col1], name=var1_dd.value,
                marker_color=PALETTE["primary"],
            ))

    elif chart == "Violin Plot":
        if split:
            for sp in _species_list:
                traces.append(go.Violin(
                    y=_iris_df[_iris_df["species"] == sp][col1],
                    name=sp, fillcolor=_sp_colors[sp], line_color=_sp_colors[sp],
                    opacity=0.7, box_visible=True, meanline_visible=True,
                ))
        else:
            traces.append(go.Violin(
                y=_iris_df[col1], name=var1_dd.value,
                fillcolor=PALETTE["primary"], line_color=PALETTE["primary"],
                opacity=0.7, box_visible=True, meanline_visible=True,
            ))

    elif chart == "Scatter":
        if split:
            for sp in _species_list:
                sub = _iris_df[_iris_df["species"] == sp]
                traces.append(go.Scatter(
                    x=sub[col1], y=sub[col2], mode="markers", name=sp,
                    marker=dict(color=_sp_colors[sp], size=8, opacity=0.7),
                ))
        else:
            traces.append(go.Scatter(
                x=_iris_df[col1], y=_iris_df[col2], mode="markers",
                name=f"{var1_dd.value} vs {var2_dd.value}",
                marker=dict(color=PALETTE["primary"], size=8, opacity=0.7),
            ))

    title_parts = {
        "Histogram":   f"{var1_dd.value} — Histogram",
        "Box Plot":    f"{var1_dd.value} — Box Plot",
        "Violin Plot": f"{var1_dd.value} — Violin Plot",
        "Scatter":     f"{var1_dd.value} vs {var2_dd.value} — Scatter",
    }
    x_title = "" if chart in ("Box Plot", "Violin Plot") else var1_dd.value
    y_title = var2_dd.value if chart == "Scatter" else var1_dd.value

    fig_layout = base_layout(
        title=title_parts[chart],
        xaxis_title=x_title,
        yaxis_title=y_title,
    )
    if chart == "Histogram":
        fig_layout.update(barmode="overlay")

    fig = go.Figure(data=traces, layout=fig_layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

# ── Observers ─────────────────────────────────────────────────────────────────
def _on_chart_change(change):
    var2_dd.layout.display = "flex" if chart_dd.value == "Scatter" else "none"
    render()

var1_dd.observe(render, names="value")
chart_dd.observe(_on_chart_change, names="value")
species_dd.observe(render, names="value")
var2_dd.observe(render, names="value")

# ── Display ───────────────────────────────────────────────────────────────────
panel = VBox([HBox([var1_dd, chart_dd, species_dd]), var2_dd, out])
display(panel)
render()


---
## What's happening?

Each chart type answers a different statistical question. Choosing the wrong one doesn't give you wrong data, it just hides the pattern you are trying to see.

| Chart type | Best question it answers | Key statistic it shows |
|-----------|--------------------------|----------------------|
| Histogram | What is the shape of this distribution? | Skew, modality, spread |
| Box plot | How do groups compare in spread and outliers? | Median, IQR, outliers |
| Violin plot | What is the full shape *and* the quartiles? | KDE + box plot combined |
| Scatter plot | How do two variables relate? | Correlation direction/strength |
| Heatmap | Which pairs of variables are most correlated? | Full correlation matrix |

### Anscombe's Quartet: why you must always plot your data

Four datasets can share the exact same mean, variance, and correlation (r = 0.816) while looking completely different when plotted: one linear, one curved, one with an outlier, one with a vertical cluster. The lesson: **always visualize before modeling**.

The widget above lets you explore this directly, look back at it and compare how the box plot and histogram each reveal different features of the same variable.

In [3]:
from plotly.subplots import make_subplots

# ── Anscombe's Quartet ────────────────────────────────────────────────────────
_AQ = [
    ("I — Linear",
     [10,8,13,9,11,14,6,4,12,7,5],
     [8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]),
    ("II — Curved",
     [10,8,13,9,11,14,6,4,12,7,5],
     [9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]),
    ("III — Outlier",
     [10,8,13,9,11,14,6,4,12,7,5],
     [7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]),
    ("IV — Vertical cluster",
     [8,8,8,8,8,8,8,19,8,8,8],
     [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]),
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[d[0] for d in _AQ],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

for idx, (label, xs, ys) in enumerate(_AQ):
    row, col = idx // 2 + 1, idx % 2 + 1
    x = np.array(xs, dtype=float)
    y = np.array(ys, dtype=float)

    slope, intercept, r_val, *_ = stats.linregress(x, y)
    x_line = np.array([x.min() - 0.5, x.max() + 0.5])
    y_line = slope * x_line + intercept

    fig.add_trace(go.Scatter(
        x=x, y=y, mode="markers",
        marker=dict(color=PALETTE["primary"], size=10, opacity=0.85,
                    line=dict(color="white", width=1)),
        showlegend=False,
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=x_line, y=y_line, mode="lines",
        line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        showlegend=False,
    ), row=row, col=col)

    ax_num = "" if idx == 0 else idx + 1
    fig.add_annotation(
        text=(f"x̄={x.mean():.2f}  ȳ={y.mean():.2f}<br>"
              f"s²(x)={x.var(ddof=1):.2f}  r={r_val:.3f}"),
        xref=f"x{ax_num} domain", yref=f"y{ax_num} domain",
        x=0.03, y=0.97,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=11, family=FONT["family"], color="#555"),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#ddd", borderwidth=1, borderpad=4,
    )

layout = base_layout(
    title="Anscombe's Quartet — same statistics, completely different structure",
    xaxis_title="",
    yaxis_title="",
)
layout.update(height=560, margin=dict(t=80, b=40))
fig.update_layout(layout)

for i in range(1, 5):
    n = "" if i == 1 else i
    fig.update_layout({
        f"xaxis{n}": dict(range=[0, 22], gridcolor="#eee", zeroline=False),
        f"yaxis{n}": dict(range=[2, 14], gridcolor="#eee", zeroline=False),
    })

fig.show()


---
## Real-world example: Iris flower measurements

The Iris dataset is the "hello world" of data science, four measurements on three species of iris flower. It is small enough to understand completely, yet rich enough to demonstrate almost every visualization technique.

The chart below shows a correlation heatmap of the four numeric features. Notice:

- **Notice:** Petal length and petal width are very strongly correlated (r ≈ 0.96), they almost always grow together, so including both in a model adds little new information
- **Notice:** Sepal width is weakly or negatively correlated with the other three, it behaves differently from the rest
- **Notice:** The diagonal is always 1.0 (every variable is perfectly correlated with itself), this anchors your reading of the off-diagonal values

> **Discussion question:** If you had to drop one feature from the Iris dataset to reduce redundancy, which would you choose based on this heatmap, and what information would you lose?

In [4]:
# ── Iris dataset correlation heatmap ──────────────────────────────────────────
df = px.data.iris().drop(columns=["species_id", "species"])
df.columns = ["Sepal Length", "Sepal Width", "Petal Length", "Petal Width"]

corr = df.corr().round(2)
cols = corr.columns.tolist()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=cols, y=cols,
    colorscale="RdBu",
    zmid=0,
    text=corr.values,
    texttemplate="%{text}",
    colorbar=dict(title="r"),
))
layout = base_layout(
    title="Iris Dataset — Pearson Correlation Heatmap",
    xaxis_title="",
    yaxis_title="",
)
fig.update_layout(layout)
fig.show()

### Chart type quick-reference guide

| Goal | Best chart | Watch out for |
|------|-----------|--------------|
| Show one variable's distribution | Histogram or KDE | Bin width can hide or exaggerate patterns |
| Compare distributions across groups | Box plot or violin plot | Box plot hides bimodality |
| Show relationship between two numeric variables | Scatter plot | Overplotting when n is large: use opacity |
| Show correlations across many variables | Heatmap | Color scales that mislead: always anchor at zero |
| Show a variable over time | Line chart | Smoothing can hide important spikes |

---
## Key takeaway

> **The right chart makes a pattern visible in seconds; the wrong chart hides it indefinitely — choosing the visualization is as analytical a decision as choosing the statistic.**

---
*Next up: Discrete Random Variables — moving from data we have collected to variables whose outcomes we can model probabilistically*